<a href="https://colab.research.google.com/github/KCDAS35/WEB/blob/main/demo/omnivoice_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OmniVoice-TTS Colab — T4 Quickstart

**Features:** Zero-shot voice cloning · Cross-lingual TTS · Emotional tags · Advanced inference · Voice Design

> **Requirements:** T4 GPU runtime · ~4 GB VRAM · Model download ~4 GB

## Step 1: GPU Check & Environment Setup

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU detected: {gpu} ({vram:.1f} GB VRAM)')
    if vram < 4:
        print('⚠️  WARNING: Less than 4 GB VRAM. Short outputs only — longer text may fail.')
    elif vram >= 8:
        print('✅ 8+ GB VRAM — full feature set available including longer outputs.')
else:
    print("""
    ⚠️ WARNING: No GPU detected.

    OmniVoice requires a GPU. To change runtime:
        1. Runtime → Change runtime type
        2. Select T4 GPU
        3. Save and reconnect
    """)

## Step 2: Install Dependencies

In [ ]:
# Clone OmniVoice-TTS repository
![ -d /content/OmniVoice-TTS ] || git clone --quiet --depth 1 https://github.com/AI-Quest/OmniVoice-TTS.git /content/OmniVoice-TTS
print('✅ Cloned AI-Quest/OmniVoice-TTS')

# Install dependencies
%cd /content/OmniVoice-TTS
!pip install -q -r requirements.txt
print('✅ Dependencies installed')

# Install cloudflared for public URL
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
print('✅ Cloudflared ready')

## Step 3: Download Model (~4 GB)

> The main weight is a **2.45 GB `.safetensors` checkpoint** from Hugging Face.
> If download stalls past 2 minutes, run the login cell below.

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id='AI-Quest/OmniVoice',
    local_dir='/content/models/OmniVoice',
    ignore_patterns=['*.md', '*.txt']
)
print('✅ OmniVoice model downloaded to /content/models/OmniVoice')

In [ ]:
# [Optional] Run this only if the download above stalls
from huggingface_hub import login
login()  # Paste your HF token from https://huggingface.co/settings/tokens

## Step 4: Launch Gradio Interface

The interface includes:
- **Zero-Shot Voice Cloning** — upload a 3-second audio sample
- **Cross-Lingual TTS** — clone a voice in one language, speak in another (great for English → Spanish)
- **Emotional Tags** — embed `[laughter]`, `[sigh]`, `[gasps]` in your text prompts
- **Advanced Settings** — Inference Steps (default 32), Guidance Scale, Denoising Ratio, Speed
- **Voice Design** — build from scratch: Gender, Age, Pitch, Style, Accent dropdowns

> **Tip:** Leave language on **Auto** for best cross-lingual results.
> 
> **Emotional tag placement:** Put tags at natural pauses, e.g.:
> `"I can't believe it [gasps] — that's incredible!"`

In [ ]:
import subprocess, re, threading, time

gradio_proc = subprocess.Popen(
    'python /content/OmniVoice-TTS/app.py '
    '--model_path /content/models/OmniVoice '
    '--server_port 7860 '
    '--share False',
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

cf_proc = subprocess.Popen(
    '/content/cloudflared tunnel --url http://localhost:7860 --no-autoupdate',
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

public_url = None
url_pattern = re.compile(r'(https://[a-z0-9-]+\.trycloudflare\.com)')

def watch_gradio():
    for line in gradio_proc.stdout:
        print('[gradio]', line.strip())

def watch_cf():
    global public_url
    for line in cf_proc.stdout:
        m = url_pattern.search(line)
        if m:
            public_url = m.group(1)
            print(f'\n✅ OmniVoice Public URL: {public_url}\n')
            print('Share this link — it expires when the Colab session ends.')

threading.Thread(target=watch_gradio, daemon=True).start()
threading.Thread(target=watch_cf, daemon=True).start()

print('Launching OmniVoice Gradio interface... (wait ~30 seconds)')
while not public_url:
    time.sleep(0.5)

## Tips & Feature Guide

### Zero-Shot Voice Cloning
- Upload **any audio sample as short as 3 seconds** — the model clones the voice automatically.
- Background noise is stripped automatically. Cleaner audio = better results.
- Leave language on **Auto** for best results.

### Cross-Lingual TTS (Perfect for Lima Clients!)
- Record yourself (or a client) speaking English → output in Spanish while keeping the same voice.
- Test Spanish fidelity: upload an English voice sample, type Spanish text, listen to the result.

### Emotional Tags
Add these inline in your text:
```
[laughter]  →  natural laugh inserted at that point
[sigh]      →  sigh/exhale
[gasps]     →  sharp intake of breath
```
**Example:** `"Bienvenido a nuestra clínica dental [laughter] — ¡nos alegra tenerte aquí!"`

### Advanced Inference Settings
| Setting | Default | Effect |
|---------|---------|--------|
| Inference Steps | 32 | Higher = better quality, slower |
| Guidance Scale | ~3.0 | Higher = closer to reference voice |
| Denoising Ratio | auto | Handles noisy source audio |
| Speed | 1.0 | Slow down / speed up output |

### Voice Design (Experimental)
- Dropdowns: **Gender, Age Group, Pitch, Style, Accent**
- ⚠️ Currently inconsistent (Male may produce Female voice) — check for model updates.

### Micro-Pauses for Human-Sounding Output
Use commas and ellipses to create natural pauses:
```
"Hola... bienvenido a Multiservicios ORE, ¿en qué le podemos ayudar?"
```